# 08a — Análise secundária: ARIMA/SARIMA(X) e Prophet

**Lê:** série completa (trava, finalidade *secundaria*), `exogenas_semanais`, `previsoes_teste`.  
**Grava:** `secundaria_pred_vendedor_estat`, `secundaria_pred_agregado`.

Não participa da seleção. Ordens escolhidas por AIC **no treino**; previsão de 1 passo no teste. Por vendedor: parâmetros do treino aplicados com filtro de Kalman (sem reestimar). Agregado: reestimado a cada semana.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "08a_secundaria_estatisticos"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import secundaria as S

spec_json = A.carregar_json("painel_spec")
TEST_END = pd.Timestamp(A.carregar_json("corte")["TEST_END"])
exog = A.carregar_df("exogenas_semanais")
serie = A.carregar_serie_completa(finalidade="secundaria")
comparison = A.carregar_df("previsoes_teste")
test_weeks = np.sort(comparison["target_week"].unique())

MACRO = spec_json["MACRO_FEATURES"]
sec_exog, EXOG_COLS, faixa = S.preparar_exogenas(exog, MACRO, C.SEC_EXOG_CAL, C.PANEL_START, C.TRAIN_TARGET_END, TEST_END)
assert pd.DatetimeIndex(np.sort(serie["week_start"].unique())).isin(sec_exog.index).all()
A.tabela(faixa, "secundaria_faixa_macro.csv", index=True); display(faixa)

In [ ]:
par = S.parametros(C, EXOG_COLS)
ts_pred, specs, spec_resumo = S.rodar_arima_vendedores(serie, sec_exog, test_weeks, C.TRAIN_TARGET_END, par, C.SEC_N_JOBS)
A.salvar_df(ts_pred, "secundaria_pred_vendedor_estat")
A.tabela(specs, "secundaria_especificacoes_vendedor.csv"); A.tabela(spec_resumo, "secundaria_especificacoes_resumo.csv")
display(spec_resumo)

In [ ]:
agg_pred, agg_series, info = S.rodar_agregado(serie, sec_exog, EXOG_COLS, test_weeks, C.TRAIN_TARGET_END, C)
A.salvar_df(agg_pred, "secundaria_pred_agregado")
A.salvar_json({"agregado": info, "parametros": par}, "secundaria_configuracao")
print(info)

In [ ]:
A.finalizar()